In [ ]:
# Author: M. Riley Owens (GitHub: mrileyowens)


In [ ]:
import sys

import os
import glob

import h5py

import numpy as np

from astropy.io import fits
import astropy.units as u
from astropy.coordinates import SkyCoord

from grizli import utils

import matplotlib.pyplot as plt
from matplotlib.offsetbox import AnchoredText

sys.path.append(os.path.abspath('..'))

from mrileyowens.stats import weighted_quantile

In [ ]:
# Set common directories
home = os.getcwd()
data = f'{home}/data'
figs = f'{home}/figs'
results = f'{home}/results'

def select():

    '''
    Plot the SFHs of the 2CSFH BEAGLE models
    '''

    # Set common directories
    #home = os.getcwd()
    #data = f'{home}/data'
    #figs = f'{home}/figs'
    #results = f'{home}/results'

    files = glob.glob(f'{results}/ew/e24_f775w_dropouts_2csfh_no_lya_ews_[!m_uv_e24]*.h5')

    hdul = fits.open(f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits')

    # Make an empty list to contain the indices and coordinates of selected young, weak emission line sources
    idx_e24, coords_e24 = [], []

    # For each BEAGLE fit results file
    for i, file in enumerate(files):

        with h5py.File(file, 'r') as f:

            for j, id in enumerate(list(f.keys())):

                probs = f[id]['probabilities'][:]

                ews_o_iii, ews_h_alpha, ews_h_beta = f[id]['o_iii_ews'][:], f[id]['h_alpha_ews'][:], f[id]['h_beta_ews'][:]

                p84_h_alpha = weighted_quantile(ews_h_alpha, probs, 0.84)
                p84_o_iii_h_beta = weighted_quantile(ews_o_iii + ews_h_beta, probs, 0.84)

                if p84_h_alpha < 800 and p84_o_iii_h_beta < 800:

                    idx = np.where(hdul[1].data['ID'] == id)

                    f150w = (hdul[1].data['NRC_F150W'] * u.nJy).to(u.ABmag)[idx]
                    f200w = (hdul[1].data['NRC_F200W'] * u.nJy).to(u.ABmag)[idx]
                    f277w = (hdul[1].data['NRC_F277W'] * u.nJy).to(u.ABmag)[idx]

                    color = (f200w - f277w) - (f150w - f200w)

                    if color < 0.3:

                        # Add the object's coordinates to the list of E24 young, weak emission line sources
                        idx_e24.append(idx[0][0])
                        coords_e24.append([hdul[1].data['RA'][idx][0], hdul[1].data['DEC'][idx][0]])

    # Convert the coordinate list to a NumPy array
    idx_e24, coords_e24 = np.array(idx_e24, dtype=np.int64), np.array(coords_e24, dtype=np.float64)

    ids = hdul[1].data['ID'][idx_e24]

    np.savetxt(f'{results}/e24_young_weak_emission_line_ids.txt', ids, fmt='%s')

def f200w():

    '''
    Plot the F200W distribution of the young, weak emission line sources
    '''

    # Get the E24 IDs of the young, weak emission line sources
    ids = np.loadtxt(f'{results}/e24_young_weak_emission_line_ids.txt', dtype=str)

    # Open the HDU list of the E24 catalog
    hdul = fits.open(f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits')

    # Get the indices of the young, weak emission line sources in the E24 catalog
    idx = np.array([np.where(hdul[1].data['ID'] == id)[0][0] for id in ids], dtype=np.int64)

    # Get the F200W photometry of the young, weak emission line sources
    f200w = (hdul[1].data['NRC_F200W'][idx] * u.nJy).to(u.ABmag)

    # Make a new figure to plot the F200W distribution of the young, weak emission line sources
    fig, ax = plt.subplots()

    # Plot the F200W histogram
    ax.hist(f200w.value, bins=20)

    # Label the axes
    ax.set_xlabel('F200W (AB mag.) (E24)')
    ax.set_ylabel('Count')

    # Save the figure
    fig.savefig(f'{figs}/e24_young_weak_emission_line_f200w.png', bbox_inches='tight', dpi=200)

    plt.close('all')

def download_dja_spectra():

    # Get the E24 IDs of the young, weak emission line sources
    ids_e24 = np.loadtxt(f'{results}/e24_young_weak_emission_line_ids.txt', dtype=str)

    # Open the HDU list of the E24 catalog
    hdul_e24 = fits.open(f'{data}/JADES_z6to9LBGcatalog_Endsley2024_f775w_dropouts.fits')

    # Get the indices of the young, weak emission line sources in the E24 catalog
    idx_e24 = np.array([np.where(hdul_e24[1].data['ID'] == id)[0][0] for id in ids_e24], dtype=np.int64)

    # Get the coordinates of the sources as a SkyCoord object
    coords_e24 = SkyCoord(ra=hdul_e24[1].data['RA'][idx_e24] * u.deg, dec=hdul_e24[1].data['DEC'][idx_e24] * u.deg)

    # -----------------------------------------------
    # Get the DJA spectroscopic catalog's coordinates
    # -----------------------------------------------

    # Get the roots, file names, and coordinates of all the DJA spectroscopic targets
    roots, files, ra, dec, ids_dja, z = np.loadtxt(f'{data}/dja_nirspec_public_v4.4.csv', dtype=str, delimiter=',', skiprows=1, usecols=(0,1,2,3,4,17), unpack=True)

    # Drop the quotation marks around each entry
    roots, files, ra, dec, ids_dja, z = np.char.replace(roots, '"', ''), np.char.replace(files, '"', ''), np.char.replace(ra, '"', ''), np.char.replace(dec, '"', ''), np.char.replace(ids_dja, '"', ''), np.char.replace(z, '"', '')

    # Convert the coordinate arrays to floats and imbue them with units
    ra_deg, dec_deg = ra.astype(np.float64) * u.deg, dec.astype(np.float64) * u.deg

    # Assemble the catalog's coordinates as a SkyCoord object
    coords_dja = SkyCoord(ra=ra_deg, dec=dec_deg)

    # Find the best matches of the E24 young, weak emission line objects in the spectroscopic DJA catalog
    idx_dja, d2d, _ = coords_e24.match_to_catalog_sky(coords_dja)

    # Set the maximum angular separation between two objects to qualify as a coordinate match
    max_sep = 0.1 * u.arcsec

    # Make a mask of the pairs that are coordinate matches
    mask = d2d < max_sep

    # Get just the indices with a coordinate match
    idx_matches = idx_dja[mask]

    # For the DJA file name of each coordinate match
    for i, _ in enumerate(files[idx_matches]):

        # Get the DJA source ID of the object
        id_dja = ids_dja[idx_matches][i]
        root = roots[idx_matches][i]

        # Get the DJA files of all the object's spectra (those with a matching root and source ID)
        files_id = files[(ids_dja == id_dja) & (roots == root) & ['prism' in j for j in files]]

        # Make a new figure to plot the spectra associated with the source
        fig, ax = plt.subplots()

        # For each DJA file with a matching source ID
        for j, file in enumerate(files_id):

            # Construct the URL to the file
            url = f'https://s3.amazonaws.com/msaexp-nirspec/extractions/{root}/{file}'

            response = requests.get(url)

            with open(f'{data}/dja_young_weak_line_spectra/{file}', 'wb') as f:
                f.write(response.content)

        '''
            spec = msaexp.spectrum.SpectrumSampler(url)

            w_obs_um = spec.spec_wobs * u.um
            f_nu_obs_njy = (spec.spec['flux'] * u.uJy).to(u.nJy)# / (1 + 6)

            mask_p95 = (f_nu_obs_njy <= np.percentile(f_nu_obs_njy, 95))

            mask_f = f_nu_obs_njy != 0

            ax.plot(w_obs_um[mask_f], f_nu_obs_njy[mask_f], ds='steps-mid', alpha=0.5, lw=0.5)

        ax.set_xlabel('Observed wavelength ($\mu$m)')
        ax.set_ylabel('Flux density (nJy)')

        ax.set_ylim(0, 20 * np.median(np.abs(f_nu_obs_njy)[mask_f]).value)

        at = AnchoredText(f'{hdul[1].data['ID'][idx_e24[mask]][i]}\n$z={z[idx_matches][i]}$', loc='upper left')
        ax.add_artist(at)
        '''

def plot():

    files = glob.glob(f'{data}/dja_young_weak_line_spectra/*.spec.fits')

    tab = utils.read_catalog(f'{data}/dja_msaexp_emission_lines_v4.4.csv', format='csv')

    for i, file in enumerate(files):

        #print(fits.open(file).info())

        hdul = fits.open(file)

        #print(repr(hdul[2].header))
        #print(hdul[1].columns)

        w_um = hdul[1].data['wave'] * u.um
        f_ujy = hdul[1].data['flux'] * u.uJy
        f_err_ujy = hdul[1].data['err'] * u.uJy

        #url = f'https://s3.amazonaws.com/msaexp-nirspec/extractions/{os.path.basename(file).split('_')[0]}/{os.path.basename(file)}'

        #tab = utils.read_catalog(f'{data}/dja_msaexp_emission_lines_v4.4.csv', format='csv') 

        #print(tab['z_best'][tab['file'] == os.path.basename(file)].data[0])

        z = tab['z_best'][tab['file'] == os.path.basename(file)].data[0]

        #ax_rest = ax.secondary_xaxis('top', functions=(lambda w: w / (1 + z), lambda w: w * (1 + z)))

        #spec = msaexp.spectrum.SpectrumSampler(url)

        #z = spec.spec['zfit']
        #print(spec.spec.columns)
        #print(z)

        fig, ax = plt.subplots()

        ax_rest = ax.secondary_xaxis('top', functions=(lambda w: w / (1 + z), lambda w: w * (1 + z)))

        ax.plot(w_um, f_ujy, ds='steps-mid', c='black')

        ax.fill_between(w_um, f_ujy - f_err_ujy, f_ujy + f_err_ujy, step='mid', alpha=0.2, color='black')

        ax.set_xlabel('Observed wavelength ($\mu$m)')
        ax_rest.set_xlabel('Rest wavelength ($\mu$m)')
        ax.set_ylabel('Flux density (uJy)')

        print(np.median(np.abs(f_ujy)))

        ax.set_ylim(bottom=0, top=np.min([np.nanmax(f_ujy.value), 20 * np.median(np.abs(f_ujy[~np.isnan(f_ujy)]).value)]))

def balmer():

    files = glob.glob(f'{data}/dja_young_weak_line_spectra/*.spec.fits')

    tab = utils.read_catalog(f'{data}/dja_msaexp_emission_lines_v4.4.csv', format='csv')

    with h5py.File(f'{results}/e24_young_weak_emission_line_bbs.h5', 'w') as f:

        for i, file in enumerate(files):

            hdul = fits.open(file)

            w_um = hdul[1].data['wave'] * u.um
            f_ujy = hdul[1].data['flux'] * u.uJy
            f_err_ujy = hdul[1].data['err'] * u.uJy

            z = tab['z_best'][tab['file'] == os.path.basename(file)].data[0]

            f_mc_ujy = np.random.normal(loc=f_ujy, scale=f_err_ujy, size=(1000, len(f_ujy)))

            bb_ratios = []

            n = 0

            #print(file)

            while n != (len(f_mc_ujy) - 1):

                #mask = f_mc_ujy[n] <= 0.
                
                #f_mc_ujy[n] = np.where(mask, 0, f_mc_ujy[n])

                bb_ratio = np.interp(0.42 * (1 + z) * u.um, w_um, f_mc_ujy[n]) / np.interp(0.35 * (1 + z) * u.um, w_um, f_mc_ujy[n])
                #print(bb_ratio)

                if np.isnan(bb_ratio):

                    #print(n, np.interp(0.42 * (1 + z) * u.um, w_um, f_mc_ujy[n]), np.interp(0.35 * (1 + z) * u.um, w_um, f_mc_ujy[n]))

                    f_mc_ujy[n] = np.random.normal(loc=f_ujy, scale=f_err_ujy, size=len(f_ujy))

                    continue

                bb_ratios.append(bb_ratio)

                n += 1

            print(f'{np.median(bb_ratios):.2f}_-{(np.median(bb_ratios) - np.percentile(bb_ratios, 16)):.2f}^+{(np.percentile(bb_ratios, 84) - np.median(bb_ratios)):.2f}')


In [ ]:
select()

In [ ]:
f200w()

In [ ]:
download_dja_spectra()

In [ ]:
plot()

In [ ]:
balmer()